In [2]:
import os
import time
import logging
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import xgboost as xgb
except ImportError:
    xgb = None

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")

class DataFetcher:
    WB_BASE = "http://api.worldbank.org/v2"
    WDI_DEFAULT_INDICATORS = {
        "gdp_growth": "NY.GDP.MKTP.KD.ZG",
        "inflation": "FP.CPI.TOTL.ZG",
        "fdi_gdp": "BX.KLT.DINV.WD.GD.ZS",
        "investment_gdp": "NE.GDI.TOTL.ZS",
    }

    @staticmethod
    def fetch_world_bank_indicator(country_codes, indicator, start_year, end_year, sleep=0.2):
        records = []
        for c in country_codes:
            page = 1
            while True:
                url = f"{DataFetcher.WB_BASE}/country/{c}/indicator/{indicator}?format=json&per_page=1000&page={page}&date={start_year}:{end_year}"
                resp = requests.get(url)
                if resp.status_code != 200:
                    break
                payload = resp.json()
                if not isinstance(payload, list) or len(payload) < 2:
                    break
                meta, data = payload[0], payload[1]
                if not data:
                    break
                for obs in data:
                    if obs.get("value") is not None:
                        records.append({
                            "country": obs["country"]["value"],
                            "countryiso3code": obs["countryiso3code"],
                            "year": int(obs["date"]),
                            "value": float(obs["value"]),
                        })
                if page >= meta.get("pages", 1):
                    break
                page += 1
                time.sleep(sleep)
        if not records:
            return pd.DataFrame(columns=["country", "countryiso3code", "year", "value"])
        return pd.DataFrame(records)

    @staticmethod
    def build_wdi_panel(country_codes, start_year, end_year, indicators=None):
        if indicators is None:
            indicators = DataFetcher.WDI_DEFAULT_INDICATORS
        dfs = []
        for feature_name, indicator_id in indicators.items():
            df_i = DataFetcher.fetch_world_bank_indicator(country_codes, indicator_id, start_year, end_year)
            df_i = df_i.rename(columns={"value": feature_name})
            dfs.append(df_i)
        base = dfs[0]
        for df_i in dfs[1:]:
            base = base.merge(df_i[["countryiso3code", "year", df_i.columns[-1]]], on=["countryiso3code", "year"], how="outer")
        return base.sort_values(["countryiso3code", "year"]).reset_index(drop=True)

class IMFWEODataLoader:
    CSV_PATH = '/content/dataset_2026-04-04T06_50_55.739545025Z_DEFAULT_INTEGRATION_IMF.RES_WEO_9.0.0.csv'
    _df = None

    @classmethod
    def fetch_weo_series(cls, country_code, indicator_code="NGDP_RPCH", frequency="A"):
        if cls._df is None:
            cls._df = pd.read_csv(cls.CSV_PATH, low_memory=False)
        series_code = f"{country_code}.{indicator_code}.{frequency}"
        row = cls._df[cls._df['SERIES_CODE'] == series_code]
        if row.empty:
            return pd.DataFrame(columns=["year", "value"])
        year_cols = [str(y) for y in range(1980, 2031) if str(y) in row.columns]
        melted = row.melt(id_vars=['SERIES_CODE'], value_vars=year_cols, var_name='year', value_name='value')
        melted['year'] = melted['year'].astype(int)
        melted['value'] = pd.to_numeric(melted['value'], errors='coerce')
        melted = melted.dropna(subset=['value'])
        return melted[['year', 'value']].sort_values('year').reset_index(drop=True)

class FeatureEngineer:
    @staticmethod
    def make_supervised_panel(panel, target_col="gdp_growth", lags=1, dropna_target=True):
        df = panel.sort_values(["countryiso3code", "year"]).copy()
        base_exclude = {"country", "countryiso3code", "year", target_col}
        cols_to_lag = [c for c in df.columns if c not in base_exclude]
        for col in cols_to_lag:
            for lag in range(1, lags + 1):
                df[f"{col}_lag{lag}"] = df.groupby("countryiso3code")[col].shift(lag)
        df["target_next_year"] = df.groupby("countryiso3code")[target_col].shift(-1)
        feature_cols = [c for c in df.columns if c not in ("country", "countryiso3code", "year", target_col, "target_next_year")]
        if dropna_target:
            df_ml = df.dropna(subset=["target_next_year"] + feature_cols).reset_index(drop=True)
        else:
            df_ml = df.dropna(subset=feature_cols).reset_index(drop=True)
        return df_ml, feature_cols

class XGBoostEconomicModel:
    def __init__(self):
        self.model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
    def train(self, df_ml, feature_cols):
        X_train = df_ml[feature_cols].values
        y_train = df_ml["target_next_year"].values
        self.model.fit(X_train, y_train)

In [ ]:
country_codes = [
    "USA", "CHN", "JPN", "DEU", "GBR", "FRA", "IND", "ITA", "BRA", "CAN",
    "KOR", "RUS", "MEX", "IDN", "SAU", "TUR", "AUS", "ARG", "ZAF", "NGA"
]

print("Fetching WDI data from 1990 to 2023...")
panel_all = DataFetcher.build_wdi_panel(country_codes, 1990, 2023)

print("\nLoading IMF WEO historical data for comparison...")
imf_frames = []
for iso3 in country_codes:
    df_c = IMFWEODataLoader.fetch_weo_series(iso3, "NGDP_RPCH", "A")
    if not df_c.empty:
        df_c["countryiso3code"] = iso3
        imf_frames.append(df_c)

if imf_frames:
    weo_df = pd.concat(imf_frames, ignore_index=True)
    weo_df = weo_df.rename(columns={"value": "weo_gdp_growth"})
else:
    weo_df = pd.DataFrame(columns=["year", "weo_gdp_growth", "countryiso3code"])

print("Data fetching complete!")

Fetching WDI data from 1990 to 2023...


In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub
import pandas as pd
import os

print("Downloading World Development Indicators dataset from Kaggle (umitka/world-development-indicators)...")
path = kagglehub.dataset_download("umitka/world-development-indicators")
print(f"Dataset downloaded to: {path}")

# Look for CSV files in the downloaded path
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    wdi_csv_path = os.path.join(path, csv_files[0])
    print(f"Loading data from {wdi_csv_path}...")
    wdi_df = pd.read_csv(wdi_csv_path, low_memory=False)
    display(wdi_df.head())
else:
    print("No CSV file found in the downloaded dataset directory.")
    print("Files found:", os.listdir(path))

In [ ]:
print("Merging World Bank WDI and IMF WEO datasets...")

if 'panel_all' in locals() and 'weo_df' in locals():
    merged_df = pd.merge(panel_all, weo_df, on=['countryiso3code', 'year'], how='outer')
    merged_df = merged_df.sort_values(['countryiso3code', 'year']).reset_index(drop=True)

    print(f"Merged dataset shape: {merged_df.shape}\n")
    display(merged_df.head())
else:
    print("The datasets 'panel_all' and 'weo_df' are not currently in memory. Please ensure the previous cells have been executed.")

In [ ]:
if 'merged_df' in locals():
    print(f"Original shape: {merged_df.shape}")
    # Drop rows where the target variable 'gdp_growth' is missing
    merged_df = merged_df.dropna(subset=['gdp_growth']).reset_index(drop=True)

    print(f"Shape after dropping missing targets: {merged_df.shape}\n")
    display(merged_df.head())
else:
    print("The dataset 'merged_df' is not currently in memory. Please run the merge cell first.")

In [ ]:
print("Building supervised ML panel...")
df_ml_all, feature_cols = FeatureEngineer.make_supervised_panel(panel_all, dropna_target=False)

# Define fixed periods
train_start, train_end = 1990, 2015
val_start, val_end = 2016, 2019
test_start, test_end = 2020, 2022

print(f"Train period: {train_start}-{train_end}")
print(f"Validation period: {val_start}-{val_end}")
print(f"Test period (Feature Years): {test_start}-{test_end}")

train_df = df_ml_all[(df_ml_all['year'] >= train_start) & (df_ml_all['year'] <= train_end)].dropna(subset=["target_next_year"] + feature_cols)
val_df = df_ml_all[(df_ml_all['year'] >= val_start) & (df_ml_all['year'] <= val_end)].dropna(subset=["target_next_year"] + feature_cols)

model = XGBoostEconomicModel()

# Extract features and targets
X_train, y_train = train_df[feature_cols].values, train_df["target_next_year"].values
X_val, y_val = val_df[feature_cols].values, val_df["target_next_year"].values

# Train with validation set
model.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Predict on all data from 2009 (to forecast 2010 onwards)
pred_start = 2009
pred_df = df_ml_all[df_ml_all['year'] >= pred_start].dropna(subset=feature_cols).copy()

# Predict
preds = model.model.predict(pred_df[feature_cols].values)

pred_df['forecast_gdp_growth'] = preds
pred_df['forecast_year'] = pred_df['year'] + 1
pred_df['actual_gdp_growth'] = pred_df['target_next_year']

results_df = pred_df[['country', 'countryiso3code', 'forecast_year', 'forecast_gdp_growth', 'actual_gdp_growth']].dropna(subset=['actual_gdp_growth'])
eval_df = results_df.merge(weo_df, left_on=["countryiso3code", "forecast_year"], right_on=["countryiso3code", "year"], how="inner")
eval_df = eval_df.dropna(subset=['actual_gdp_growth', 'forecast_gdp_growth', 'weo_gdp_growth'])

# Calculate metrics only on the out-of-sample test period
test_eval_df = eval_df[(eval_df['forecast_year'] >= test_start + 1) & (eval_df['forecast_year'] <= test_end + 1)]

ml_mae = mean_absolute_error(test_eval_df['actual_gdp_growth'], test_eval_df['forecast_gdp_growth'])
ml_rmse = np.sqrt(mean_squared_error(test_eval_df['actual_gdp_growth'], test_eval_df['forecast_gdp_growth']))
imf_mae = mean_absolute_error(test_eval_df['actual_gdp_growth'], test_eval_df['weo_gdp_growth'])
imf_rmse = np.sqrt(mean_squared_error(test_eval_df['actual_gdp_growth'], test_eval_df['weo_gdp_growth']))

print(f"\n=== OUT-OF-SAMPLE BACKTESTING RESULTS ({test_start + 1}-{test_end + 1}) ===")
print(f"XGBoost ML Model - MAE: {ml_mae:.3f} pp, RMSE: {ml_rmse:.3f} pp")
print(f"IMF WEO Historic - MAE: {imf_mae:.3f} pp, RMSE: {imf_rmse:.3f} pp")

metrics_df = pd.DataFrame({
    'Model': ['XGBoost ML', 'IMF WEO Historic', 'XGBoost ML', 'IMF WEO Historic'],
    'Metric': ['MAE', 'MAE', 'RMSE', 'RMSE'],
    'Error (Percentage Points)': [ml_mae, imf_mae, ml_rmse, imf_rmse]
})
plt.figure(figsize=(9, 6))
sns.barplot(x='Metric', y='Error (Percentage Points)', hue='Model', data=metrics_df, palette='Set2')
plt.title(f'Out-of-Sample Forecasting Errors ({test_start + 1}-{test_end + 1})')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

if 'train_df' in locals() and 'val_df' in locals() and 'df_ml_all' in locals():
    print("Training LASSO ML Model...")

    # LASSO benefits from scaled features
    lasso_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('lasso', Lasso(alpha=0.1, random_state=42))
    ])

    # Train the model (combining train and val for a slightly larger set if desired, but let's stick to train)
    lasso_pipeline.fit(X_train, y_train)

    # Predict
    lasso_preds = lasso_pipeline.predict(pred_df[feature_cols].values)

    # Create a new results DataFrame for LASSO
    lasso_pred_df = pred_df.copy()
    lasso_pred_df['lasso_forecast'] = lasso_preds
    lasso_pred_df['forecast_year'] = lasso_pred_df['year'] + 1
    lasso_pred_df['actual_gdp_growth'] = lasso_pred_df['target_next_year']

    lasso_results_df = lasso_pred_df[['country', 'countryiso3code', 'forecast_year', 'lasso_forecast', 'actual_gdp_growth']].dropna(subset=['actual_gdp_growth'])
    lasso_eval_df = lasso_results_df.merge(weo_df, left_on=["countryiso3code", "forecast_year"], right_on=["countryiso3code", "year"], how="inner")
    lasso_eval_df = lasso_eval_df.dropna(subset=['actual_gdp_growth', 'lasso_forecast', 'weo_gdp_growth'])

    # Calculate metrics on the out-of-sample test period
    lasso_test_eval_df = lasso_eval_df[(lasso_eval_df['forecast_year'] >= test_start + 1) & (lasso_eval_df['forecast_year'] <= test_end + 1)]

    lasso_mae = mean_absolute_error(lasso_test_eval_df['actual_gdp_growth'], lasso_test_eval_df['lasso_forecast'])
    lasso_rmse = np.sqrt(mean_squared_error(lasso_test_eval_df['actual_gdp_growth'], lasso_test_eval_df['lasso_forecast']))

    print(f"\n=== OUT-OF-SAMPLE BACKTESTING RESULTS (LASSO, {test_start + 1}-{test_end + 1}) ===")
    print(f"LASSO Model - MAE: {lasso_mae:.3f} pp, RMSE: {lasso_rmse:.3f} pp")

    # Compare with existing models if metrics are available
    if 'ml_mae' in locals() and 'imf_mae' in locals():
        print(f"\nFor reference:")
        print(f"XGBoost ML Model - MAE: {ml_mae:.3f} pp, RMSE: {ml_rmse:.3f} pp")
        print(f"IMF WEO Historic - MAE: {imf_mae:.3f} pp, RMSE: {imf_rmse:.3f} pp")
else:
    print("Training data not found. Please run the previous cells first.")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

if 'train_df' in locals() and 'val_df' in locals() and 'df_ml_all' in locals():
    print("Training Random Forest ML Model...")

    # Initialize the model
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)

    # Train the model
    rf_model.fit(X_train, y_train)

    # Predict
    rf_preds = rf_model.predict(pred_df[feature_cols].values)

    # Create a new results DataFrame for Random Forest
    rf_pred_df = pred_df.copy()
    rf_pred_df['rf_forecast'] = rf_preds
    rf_pred_df['forecast_year'] = rf_pred_df['year'] + 1
    rf_pred_df['actual_gdp_growth'] = rf_pred_df['target_next_year']

    rf_results_df = rf_pred_df[['country', 'countryiso3code', 'forecast_year', 'rf_forecast', 'actual_gdp_growth']].dropna(subset=['actual_gdp_growth'])
    rf_eval_df = rf_results_df.merge(weo_df, left_on=["countryiso3code", "forecast_year"], right_on=["countryiso3code", "year"], how="inner")
    rf_eval_df = rf_eval_df.dropna(subset=['actual_gdp_growth', 'rf_forecast', 'weo_gdp_growth'])

    # Calculate metrics on the out-of-sample test period
    rf_test_eval_df = rf_eval_df[(rf_eval_df['forecast_year'] >= test_start + 1) & (rf_eval_df['forecast_year'] <= test_end + 1)]

    rf_mae = mean_absolute_error(rf_test_eval_df['actual_gdp_growth'], rf_test_eval_df['rf_forecast'])
    rf_rmse = np.sqrt(mean_squared_error(rf_test_eval_df['actual_gdp_growth'], rf_test_eval_df['rf_forecast']))

    print(f"\n=== OUT-OF-SAMPLE BACKTESTING RESULTS (Random Forest, {test_start + 1}-{test_end + 1}) ===")
    print(f"Random Forest Model - MAE: {rf_mae:.3f} pp, RMSE: {rf_rmse:.3f} pp")

    # Compare with existing models if metrics are available
    print(f"\nFor reference:")
    if 'ml_mae' in locals() and 'ml_rmse' in locals():
        print(f"XGBoost ML Model - MAE: {ml_mae:.3f} pp, RMSE: {ml_rmse:.3f} pp")
    if 'lasso_mae' in locals() and 'lasso_rmse' in locals():
        print(f"LASSO Model - MAE: {lasso_mae:.3f} pp, RMSE: {lasso_rmse:.3f} pp")
    if 'imf_mae' in locals() and 'imf_rmse' in locals():
        print(f"IMF WEO Historic - MAE: {imf_mae:.3f} pp, RMSE: {imf_rmse:.3f} pp")
else:
    print("Training data not found. Please run the previous cells first.")

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

if 'train_df' in locals() and 'val_df' in locals() and 'df_ml_all' in locals():
    print("Training KNN ML Model...")

    # KNN relies on distance metrics, so feature scaling is crucial
    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=5))
    ])

    # Train the model
    knn_pipeline.fit(X_train, y_train)

    # Predict
    knn_preds = knn_pipeline.predict(pred_df[feature_cols].values)

    # Create a new results DataFrame for KNN
    knn_pred_df = pred_df.copy()
    knn_pred_df['knn_forecast'] = knn_preds
    knn_pred_df['forecast_year'] = knn_pred_df['year'] + 1
    knn_pred_df['actual_gdp_growth'] = knn_pred_df['target_next_year']

    knn_results_df = knn_pred_df[['country', 'countryiso3code', 'forecast_year', 'knn_forecast', 'actual_gdp_growth']].dropna(subset=['actual_gdp_growth'])
    knn_eval_df = knn_results_df.merge(weo_df, left_on=["countryiso3code", "forecast_year"], right_on=["countryiso3code", "year"], how="inner")
    knn_eval_df = knn_eval_df.dropna(subset=['actual_gdp_growth', 'knn_forecast', 'weo_gdp_growth'])

    # Calculate metrics on the out-of-sample test period
    knn_test_eval_df = knn_eval_df[(knn_eval_df['forecast_year'] >= test_start + 1) & (knn_eval_df['forecast_year'] <= test_end + 1)]

    knn_mae = mean_absolute_error(knn_test_eval_df['actual_gdp_growth'], knn_test_eval_df['knn_forecast'])
    knn_rmse = np.sqrt(mean_squared_error(knn_test_eval_df['actual_gdp_growth'], knn_test_eval_df['knn_forecast']))

    print(f"\n=== OUT-OF-SAMPLE BACKTESTING RESULTS (KNN, {test_start + 1}-{test_end + 1}) ===")
    print(f"KNN Model - MAE: {knn_mae:.3f} pp, RMSE: {knn_rmse:.3f} pp")

    # Compare with existing models if metrics are available
    print(f"\nFor reference:")
    if 'ml_mae' in locals() and 'ml_rmse' in locals():
        print(f"XGBoost ML Model - MAE: {ml_mae:.3f} pp, RMSE: {ml_rmse:.3f} pp")
    if 'rf_mae' in locals() and 'rf_rmse' in locals():
        print(f"Random Forest Model - MAE: {rf_mae:.3f} pp, RMSE: {rf_rmse:.3f} pp")
    if 'lasso_mae' in locals() and 'lasso_rmse' in locals():
        print(f"LASSO Model - MAE: {lasso_mae:.3f} pp, RMSE: {lasso_rmse:.3f} pp")
    if 'imf_mae' in locals() and 'imf_rmse' in locals():
        print(f"IMF WEO Historic - MAE: {imf_mae:.3f} pp, RMSE: {imf_rmse:.3f} pp")
else:
    print("Training data not found. Please run the previous cells first.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Choose a specific country code from the available ones
target_country_iso = 'BRA' # Changed from 'JPN' to 'USA'
plot_start_year = 2010     # Show data from this year onwards

if 'eval_df' in locals() and 'merged_df' in locals():
    # Get historical actuals and IMF WEO from merged_df
    country_data = merged_df[(merged_df['countryiso3code'] == target_country_iso) & (merged_df['year'] >= plot_start_year)].sort_values('year')

    # Get ML Forecasts
    country_ml_df = eval_df[eval_df['countryiso3code'] == target_country_iso].sort_values('forecast_year')
    country_lasso_df = lasso_eval_df[lasso_eval_df['countryiso3code'] == target_country_iso].sort_values('forecast_year') if 'lasso_eval_df' in locals() else pd.DataFrame()
    country_rf_df = rf_eval_df[rf_eval_df['countryiso3code'] == target_country_iso].sort_values('forecast_year') if 'rf_eval_df' in locals() else pd.DataFrame()
    country_knn_df = knn_eval_df[knn_eval_df['countryiso3code'] == target_country_iso].sort_values('forecast_year') if 'knn_eval_df' in locals() else pd.DataFrame()

    if not country_data.empty:
        plt.figure(figsize=(14, 7))

        # Plot Actual GDP Growth from merged_df
        plt.plot(country_data['year'], country_data['gdp_growth'],
                 marker='o', linewidth=3, label='WDI Actual GDP Growth', color='black')

        # Plot IMF WEO Forecast/Historic from merged_df
        plt.plot(country_data['year'], country_data['weo_gdp_growth'],
                 marker='^', linewidth=2, linestyle=':', label='IMF WEO Historic', color='orange')

        # Plot ML Forecast
        if not country_ml_df.empty:
            plt.plot(country_ml_df['forecast_year'], country_ml_df['forecast_gdp_growth'],
                     marker='s', linewidth=2, linestyle='--', label='XGBoost Forecast', color='blue')

        if not country_lasso_df.empty:
            plt.plot(country_lasso_df['forecast_year'], country_lasso_df['lasso_forecast'],
                     marker='v', linewidth=2, linestyle='--', label='LASSO Forecast', color='green')

        if not country_rf_df.empty:
            plt.plot(country_rf_df['forecast_year'], country_rf_df['rf_forecast'],
                     marker='D', linewidth=2, linestyle='--', label='Random Forest Forecast', color='red')

        if not country_knn_df.empty:
            plt.plot(country_knn_df['forecast_year'], country_knn_df['knn_forecast'],
                     marker='P', linewidth=2, linestyle='--', label='KNN Forecast', color='purple')

        plt.title(f'Actual vs Predicted GDP Growth for {target_country_iso} (from {plot_start_year})', fontsize=14)
        plt.xlabel('Year', fontsize=12)
        plt.ylabel('GDP Growth (%)', fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, linestyle='--', alpha=0.7)

        all_years = np.arange(plot_start_year, country_data['year'].max() + 1)
        plt.xticks(all_years, rotation=45)

        plt.tight_layout()
        plt.show()
    else:
        print(f"No data available for country code: {target_country_iso}")
else:
    print("Required datasets (merged_df, eval_df) are not currently in memory. Please run the previous cells first.")

In [ ]:
import numpy as np
import pandas as pd

if 'test_eval_df' in locals():
    print("Calculating Mean Absolute Error (MAE) for each country during the test period...")

    # Calculate absolute error per row
    test_eval_df = test_eval_df.copy()
    test_eval_df['abs_error'] = np.abs(test_eval_df['actual_gdp_growth'] - test_eval_df['forecast_gdp_growth'])

    # Group by country and calculate MAE
    country_errors = test_eval_df.groupby(['country', 'countryiso3code'])['abs_error'].mean().reset_index()
    country_errors = country_errors.rename(columns={'abs_error': 'MAE (Percentage Points)'})

    # Sort to get the lowest errors
    top_5_countries = country_errors.sort_values('MAE (Percentage Points)').head(5).reset_index(drop=True)

    print("\nTop 5 Countries with the Lowest XGBoost Prediction Error:")
    display(top_5_countries)

    # Optional: Plot the top 5
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(8, 5))
    sns.barplot(x='MAE (Percentage Points)', y='country', hue='country', data=top_5_countries, palette='viridis', legend=False)
    plt.title('Top 5 Countries by Lowest XGBoost MAE (2021-2023)')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("The dataset 'test_eval_df' is not currently in memory. Please run the model evaluation cell first.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if 'top_5_countries' in locals() and 'eval_df' in locals() and 'merged_df' in locals():
    top_5_isos = top_5_countries['countryiso3code'].tolist()
    plot_start_year = 2010

    fig, axes = plt.subplots(len(top_5_isos), 1, figsize=(10, 5 * len(top_5_isos)), sharex=True)

    for i, iso in enumerate(top_5_isos):
        country_name = top_5_countries.loc[i, 'country']

        # Get historical actuals and IMF WEO from merged_df
        c_merged = merged_df[(merged_df['countryiso3code'] == iso) & (merged_df['year'] >= plot_start_year)].sort_values('year')

        # Get ML Forecasts
        c_ml = eval_df[eval_df['countryiso3code'] == iso].sort_values('forecast_year')
        c_lasso = lasso_eval_df[lasso_eval_df['countryiso3code'] == iso].sort_values('forecast_year') if 'lasso_eval_df' in locals() else pd.DataFrame()
        c_rf = rf_eval_df[rf_eval_df['countryiso3code'] == iso].sort_values('forecast_year') if 'rf_eval_df' in locals() else pd.DataFrame()
        c_knn = knn_eval_df[knn_eval_df['countryiso3code'] == iso].sort_values('forecast_year') if 'knn_eval_df' in locals() else pd.DataFrame()

        ax = axes[i]

        # Trendline and scatter for Actuals (merged_df)
        if not c_merged.empty:
            ax.plot(c_merged['year'], c_merged['gdp_growth'],
                    marker='o', markersize=6, linewidth=2, label='WDI Actual GDP Growth', color='black')

            # Trendline and scatter for IMF WEO (merged_df)
            ax.plot(c_merged['year'], c_merged['weo_gdp_growth'],
                    marker='^', markersize=6, linewidth=2, linestyle=':', label='IMF WEO Historic', color='orange')

        # Trendline and scatter for ML Forecasts
        if not c_ml.empty:
            ax.plot(c_ml['forecast_year'], c_ml['forecast_gdp_growth'],
                    marker='s', markersize=6, linewidth=2, linestyle='--', label='XGBoost Forecast', color='blue')
        if not c_lasso.empty:
            ax.plot(c_lasso['forecast_year'], c_lasso['lasso_forecast'],
                    marker='v', markersize=6, linewidth=2, linestyle='--', label='LASSO Forecast', color='green')
        if not c_rf.empty:
            ax.plot(c_rf['forecast_year'], c_rf['rf_forecast'],
                    marker='D', markersize=6, linewidth=2, linestyle='--', label='Random Forest Forecast', color='red')
        if not c_knn.empty:
            ax.plot(c_knn['forecast_year'], c_knn['knn_forecast'],
                    marker='P', markersize=6, linewidth=2, linestyle='--', label='KNN Forecast', color='purple')

        ax.set_title(f'{country_name} ({iso}) - Actual vs IMF vs ML Forecasts', fontsize=12)
        ax.set_ylabel('GDP Growth (%)')
        ax.grid(True, linestyle='--', alpha=0.7)
        ax.legend(loc='lower left', fontsize=9)

    axes[-1].set_xlabel('Year', fontsize=12)

    # Set x-ticks across all years starting from plot_start_year
    if not c_merged.empty:
        all_years = np.arange(plot_start_year, c_merged['year'].max() + 1)
        plt.xticks(all_years, rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("Required datasets are missing. Please run the previous cells first.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

if 'merged_df' in locals() and 'eval_df' in locals():
    print("Comparing ML predictions independently against the WDI/IMF WEO merged dataset...\n")

    # Collect model metrics and dataframes
    models = []

    # 1. IMF
    valid_imf = merged_df.dropna(subset=['gdp_growth', 'weo_gdp_growth'])
    models.append({
        'name': 'IMF WEO Historic',
        'df': valid_imf,
        'actual_col': 'gdp_growth',
        'pred_col': 'weo_gdp_growth',
        'color': 'orange'
    })

    # 2. XGBoost
    valid_ml = eval_df.dropna(subset=['actual_gdp_growth', 'forecast_gdp_growth'])
    models.append({
        'name': 'XGBoost ML',
        'df': valid_ml,
        'actual_col': 'actual_gdp_growth',
        'pred_col': 'forecast_gdp_growth',
        'color': 'blue'
    })

    # 3. LASSO
    if 'lasso_eval_df' in locals():
        valid_lasso = lasso_eval_df.dropna(subset=['actual_gdp_growth', 'lasso_forecast'])
        models.append({
            'name': 'LASSO',
            'df': valid_lasso,
            'actual_col': 'actual_gdp_growth',
            'pred_col': 'lasso_forecast',
            'color': 'green'
        })

    # 4. Random Forest
    if 'rf_eval_df' in locals():
        valid_rf = rf_eval_df.dropna(subset=['actual_gdp_growth', 'rf_forecast'])
        models.append({
            'name': 'Random Forest',
            'df': valid_rf,
            'actual_col': 'actual_gdp_growth',
            'pred_col': 'rf_forecast',
            'color': 'red'
        })

    # 5. KNN
    if 'knn_eval_df' in locals():
        valid_knn = knn_eval_df.dropna(subset=['actual_gdp_growth', 'knn_forecast'])
        models.append({
            'name': 'KNN',
            'df': valid_knn,
            'actual_col': 'actual_gdp_growth',
            'pred_col': 'knn_forecast',
            'color': 'purple'
        })

    print("=== COMPARISON RESULTS ===")
    for m in models:
        mae = mean_absolute_error(m['df'][m['actual_col']], m['df'][m['pred_col']])
        rmse = np.sqrt(mean_squared_error(m['df'][m['actual_col']], m['df'][m['pred_col']]))
        print(f"{m['name']} (from results, N={len(m['df'])}) - MAE: {mae:.3f} pp, RMSE: {rmse:.3f} pp")
    print()

    # Visualizations
    n_models = len(models)
    cols = 3
    rows = (n_models + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten()

    for i, m in enumerate(models):
        ax = axes[i]
        df = m['df']
        sns.scatterplot(data=df, x=m['actual_col'], y=m['pred_col'], alpha=0.6, ax=ax, color=m['color'])
        min_val = min(df[m['actual_col']].min(), df[m['pred_col']].min())
        max_val = max(df[m['actual_col']].max(), df[m['pred_col']].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')
        ax.set_title(f"{m['name']} vs Actual")
        ax.set_xlabel('Actual GDP Growth (%)')
        ax.set_ylabel(f"{m['name']} Forecast (%)")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.7)

    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
else:
    print("Required datasets are not currently in memory. Please ensure previous cells have been executed.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

if 'models' in locals():
    results_list = []
    for m in models:
        df = m['df']
        mae = mean_absolute_error(df[m['actual_col']], df[m['pred_col']])
        rmse = np.sqrt(mean_squared_error(df[m['actual_col']], df[m['pred_col']]))

        results_list.append({
            'Model': m['name'],
            'MAE (pp)': round(mae, 3),
            'RMSE (pp)': round(rmse, 3),
        })

    comparison_table = pd.DataFrame(results_list)

    print("=== Overall Model Performance Comparison ===")
    # Display sorted by MAE for easy comparison
    display(comparison_table.sort_values('MAE (pp)').reset_index(drop=True))
else:
    print("The 'models' list is not available. Please run the previous visualization cell first.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'comparison_table' in locals():
    # Melt the comparison_table for easier plotting with seaborn
    plot_df = comparison_table.melt(
        id_vars=['Model'],
        value_vars=['MAE (pp)', 'RMSE (pp)'],
        var_name='Metric',
        value_name='Error (pp)'
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=plot_df, x='Model', y='Error (pp)', hue='Metric', palette='Set2')
    plt.title('Model Performance Comparison (MAE and RMSE)')
    plt.ylabel('Error (Percentage Points)')
    plt.xlabel('Model')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("The 'comparison_table' is not available. Please run the previous cell first.")

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Suppress convergence warnings from ARIMA for cleaner output
warnings.filterwarnings("ignore")

forecast_years = [2024, 2025, 2026, 2027]
base_features = ['inflation', 'fdi_gdp', 'investment_gdp']

if 'panel_all' in locals():
    country_codes = panel_all['countryiso3code'].unique()
    print("Forecasting independent features using ARIMA(1,1,0)...")

    future_records = []

    for iso in country_codes:
        c_data = panel_all[panel_all['countryiso3code'] == iso].sort_values('year')
        c_name = c_data['country'].iloc[0] if not c_data.empty else iso

        # Initialize records for this country's future years
        iso_records = [{'country': c_name, 'countryiso3code': iso, 'year': y} for y in forecast_years]

        for feat in base_features:
            # Get historical series, dropping NaNs
            series = c_data[feat].dropna().values

            if len(series) < 5:
                # Not enough data, fallback to LOCF
                preds = [series[-1]] * len(forecast_years) if len(series) > 0 else [0] * len(forecast_years)
            else:
                try:
                    # Fit a simple ARIMA(1, 1, 0) model
                    model = ARIMA(series, order=(1, 1, 0))
                    res = model.fit()
                    preds = res.forecast(steps=len(forecast_years))
                except Exception:
                    # Fallback to LOCF if ARIMA fails to converge
                    preds = [series[-1]] * len(forecast_years)

            # Map predictions to the iso_records
            for i, year in enumerate(forecast_years):
                iso_records[i][feat] = preds[i]

        future_records.extend(iso_records)

    future_features_df = pd.DataFrame(future_records)

    print("\nARIMA Forecasting complete. Sample of predicted features for the USA:")
    display(future_features_df[future_features_df['countryiso3code'] == 'USA'])

    print("\nPlotting Forecasted Features for Multiple Countries...")
    countries_to_plot = ['USA', 'CHN', 'DEU', 'JPN', 'GBR', 'IND']
    plot_df = future_features_df[future_features_df['countryiso3code'].isin(countries_to_plot)]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, feat in enumerate(base_features):
        sns.lineplot(data=plot_df, x='year', y=feat, hue='country', marker='o', ax=axes[i])
        axes[i].set_title(f'ARIMA Forecast: {feat}')
        axes[i].set_xticks(forecast_years)
        axes[i].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
else:
    print("The dataset 'panel_all' is missing. Please run previous cells.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

# Ensure models_dict is defined for the subsequent logic
# Re-instantiate the XGBoost model since the 'model' variable was overwritten by ARIMA in the previous cell
models_dict = {
    'XGBoost': XGBoostEconomicModel().model,
    'LASSO': lasso_pipeline,
    'Random Forest': rf_model,
    'KNN': knn_pipeline
}

if 'future_features_df' in locals():
    print("Applying ML models using ARIMA-forecasted features...")

    # 1. Train models on all available historical data
    train_data = df_ml_all.dropna(subset=feature_cols + ['target_next_year'])
    X_train_full = train_data[feature_cols].values
    y_train_full = train_data['target_next_year'].values

    for name, m in models_dict.items():
        m.fit(X_train_full, y_train_full)

    # 2. Combine historical base features with future ARIMA forecasts to compute lags
    hist_features = panel_all[['country', 'countryiso3code', 'year'] + base_features].copy()
    combined_features = pd.concat([hist_features, future_features_df], ignore_index=True).sort_values(['countryiso3code', 'year'])

    # Create lag 1 features
    for col in base_features:
        combined_features[f"{col}_lag1"] = combined_features.groupby("countryiso3code")[col].shift(1)

    # 3. Predict GDP growth for 2024-2027
    # (To forecast year Y, we use features from year Y-1)
    pred_years = [2023, 2024, 2025, 2026]
    dynamic_forecasts = []

    for p_year in pred_years:
        f_year = p_year + 1
        curr_data = combined_features[combined_features['year'] == p_year].dropna(subset=feature_cols)

        for _, row in curr_data.iterrows():
            res = {
                'country': row['country'],
                'countryiso3code': row['countryiso3code'],
                'forecast_year': f_year
            }
            X_pred = row[feature_cols].values.reshape(1, -1)
            for name, m in models_dict.items():
                res[name] = m.predict(X_pred)[0]
            dynamic_forecasts.append(res)

    dynamic_forecasts_df = pd.DataFrame(dynamic_forecasts)

    # 4. Plot the dynamic forecasts for a target country (e.g., USA)
    target_iso = 'USA'
    usa_dyn = dynamic_forecasts_df[dynamic_forecasts_df['countryiso3code'] == target_iso]

    if not usa_dyn.empty:
        plot_df_dyn = usa_dyn.melt(
            id_vars=['forecast_year'],
            value_vars=['XGBoost', 'LASSO', 'Random Forest', 'KNN'],
            var_name='Model',
            value_name='GDP Growth Forecast (%)'
        )

        plt.figure(figsize=(10, 6))
        sns.lineplot(data=plot_df_dyn, x='forecast_year', y='GDP Growth Forecast (%)', hue='Model', marker='o', linewidth=2.5, markersize=8)
        plt.title(f'Dynamic Multi-Model GDP Growth Forecast for {target_iso} (2024-2027)\nUsing ARIMA-Predicted Features', fontsize=14)
        plt.xlabel('Forecast Year', fontsize=12)
        plt.ylabel('GDP Growth Forecast (%)', fontsize=12)
        plt.xticks([2024, 2025, 2026, 2027])
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='Model')
        plt.tight_layout()
        plt.show()

        print(f"\nDynamic Tabular Forecasts for {target_iso}:")
        display(usa_dyn.set_index('forecast_year').drop(columns=['country', 'countryiso3code']).round(3))
    else:
        print(f"No forecasting data available for {target_iso}.")
else:
    print("Required datasets or models are not in memory. Please run the previous cells.")

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Variables Considered
variables_data = [
    {"Variable Name": "gdp_growth", "Source": "World Bank (WDI)", "Role": "Primary Target Variable", "Description": "GDP growth (annual %)"},
    {"Variable Name": "weo_gdp_growth", "Source": "IMF WEO", "Role": "Comparison / Baseline", "Description": "Gross domestic product, constant prices (Percent change)"},
    {"Variable Name": "inflation", "Source": "World Bank (WDI)", "Role": "Feature", "Description": "Inflation, consumer prices (annual %)"},
    {"Variable Name": "fdi_gdp", "Source": "World Bank (WDI)", "Role": "Feature", "Description": "Foreign direct investment, net inflows (% of GDP)"},
    {"Variable Name": "investment_gdp", "Source": "World Bank (WDI)", "Role": "Feature", "Description": "Gross capital formation (% of GDP)"},
    {"Variable Name": "inflation_lag1", "Source": "Engineered", "Role": "Feature", "Description": "1-year lag of inflation"},
    {"Variable Name": "fdi_gdp_lag1", "Source": "Engineered", "Role": "Feature", "Description": "1-year lag of FDI to GDP ratio"},
    {"Variable Name": "investment_gdp_lag1", "Source": "Engineered", "Role": "Feature", "Description": "1-year lag of Investment to GDP ratio"}
]
variables_df = pd.DataFrame(variables_data)

# 2. Countries Used
country_codes_used = [
    "USA", "CHN", "JPN", "DEU", "GBR", "FRA", "IND", "ITA", "BRA", "CAN",
    "KOR", "RUS", "MEX", "IDN", "SAU", "TUR", "AUS", "ARG", "ZAF", "NGA"
]
# Assuming panel_all is in memory, we can get their full names:
if 'panel_all' in locals():
    country_mapping = panel_all[['countryiso3code', 'country']].drop_duplicates().set_index('countryiso3code')['country'].to_dict()
    countries_data = [{"ISO3 Code": code, "Country Name": country_mapping.get(code, code)} for code in country_codes_used]
else:
    countries_data = [{"ISO3 Code": code} for code in country_codes_used]

countries_df = pd.DataFrame(countries_data)

display(Markdown("### 📊 Variables Considered"))
display(variables_df)

display(Markdown("### 🌍 Countries Analyzed"))
display(countries_df)


### ና1 Machine Learning Models Overview & Hyperparameters

Here is a detailed breakdown of what each machine learning (ML) and forecasting model is doing to predict GDP growth, along with an explanation of the specific hyperparameters used in the code:

#### 1. XGBoost (Extreme Gradient Boosting)
*   **How it works:** XGBoost is an advanced ensemble technique that builds a series of decision trees sequentially. Each new tree is specifically designed to correct the errors (residuals) made by the previous trees.
*   **Role in Notebook:** It acts as the primary predictive model, finding complex, non-linear patterns in historical economic indicators.
*   **Hyperparameters Used:**
    *   `n_estimators=100`: The model will build 100 sequential trees. More trees increase learning capacity but can lead to overfitting if too high.
    *   `learning_rate=0.05`: Also known as "eta". It shrinks the contribution of each new tree by 5%. A lower learning rate makes the model more robust and less prone to overfitting, but requires more trees (`n_estimators`) to learn effectively.
    *   `max_depth=4`: Restricts each tree to a maximum depth of 4 splits. Keeping this low (shallow trees) prevents the model from memorizing noise in the training data (overfitting).
    *   `random_state=42`: Ensures the randomness in the algorithm is reproducible, so you get the same results every time you run it.

#### 2. LASSO Regression (Least Absolute Shrinkage and Selection Operator)
*   **How it works:** LASSO is a linear regression model that includes an L1 regularization penalty. This penalty discourages overly complex models by shrinking the coefficients of less important features all the way to exactly zero.
*   **Role in Notebook:** It provides a simpler, linear baseline and performs automatic feature selection. It is paired with a `StandardScaler` because distance/regularization metrics are highly sensitive to the scale of the data.
*   **Hyperparameters Used:**
    *   `alpha=0.1`: This is the regularization strength. A value of 0.1 applies a moderate penalty. If `alpha` were 0, it would be a standard Linear Regression. Higher values force more coefficients to zero, creating a simpler model.
    *   `random_state=42`: Ensures reproducibility for any randomized underlying solver behavior.

#### 3. Random Forest
*   **How it works:** An ensemble method that builds multiple decision trees *independently* and simultaneously, using different random subsets of the data and features (bagging). The final prediction is the average of all trees.
*   **Role in Notebook:** It acts as a robust, low-variance alternative to XGBoost. It finds a stable consensus among independent models.
*   **Hyperparameters Used:**
    *   `n_estimators=100`: The model builds 100 independent trees. In Random Forests, adding more trees generally improves performance without increasing the risk of overfitting.
    *   `max_depth=4`: Restricts each tree to 4 splits deep, keeping individual models simple and highly generalized.
    *   `random_state=42`: Ensures the random sampling of data and features is consistent across runs.

#### 4. K-Nearest Neighbors (KNN)
*   **How it works:** A non-parametric algorithm that predicts GDP growth by finding the $K$ most similar historical records (neighbors) based on the input features, and averaging their GDP growth.
*   **Role in Notebook:** Provides a similarity-based forecast. Wrapped in a `Pipeline` with `StandardScaler` to ensure features with larger numeric ranges (like inflation) don't unfairly dominate the distance calculation.
*   **Hyperparameters Used:**
    *   `n_neighbors=5`: The algorithm looks at exactly the 5 closest historical data points to make its prediction. A smaller number makes the model more sensitive to local noise, while a larger number smooths out predictions but might include irrelevant "neighbors."

#### 5. ARIMA (Autoregressive Integrated Moving Average)
*   **How it works:** A classic univariate time-series model that predicts future values based entirely on its own past values, differencing, and errors.
*   **Role in Notebook:** Used as a helper model to forecast the independent features (inflation, FDI, investment) for the years 2024–2027. The ML models then use these ARIMA-predicted features to forecast future GDP.
*   **Hyperparameters Used:**
    *   `order=(1, 1, 0)`: This defines the $(p, d, q)$ parameters of the model.
        *   `p=1` (Autoregressive term): Uses the value from 1 previous time step (lag 1) to predict the current value.
        *   `d=1` (Differencing term): Applies 1st-order differencing (subtracting the previous value from the current value) once to make the time series stationary (removing trends).
        *   `q=0` (Moving Average term): Does not use past forecast errors for prediction.

In [ ]:
import pandas as pd
from google.colab import files

if 'merged_df' in locals():
    filename = 'merged_cleaned_dataset.xlsx'
    print(f"Exporting dataset to {filename}...")

    # Save the cleaned and merged DataFrame to an Excel file
    merged_df.to_excel(filename, index=False)

    print("Download should start automatically. If not, check your browser permissions.")
    # Trigger the download prompt in the browser
    files.download(filename)
else:
    print("The 'merged_df' dataset is not currently in memory. Please ensure the data merging and cleaning cells have been executed.")